# 04 — Diagnostic: tests, effect sizes, uncertainty

**Stage 5a of the Tier A pipeline (Path A).** `03` showed four patterns by eye: the pair construction,
income setting spread rather than level, an age ceiling at 40, and no gender gap. This notebook puts
each one through a test that fits its shape, reports an **effect size with a confidence interval**
alongside every p-value, and corrects for multiple comparisons (Benjamini–Hochberg) across all of them.

| § | Claim from `03` | Test | Effect size |
|---|---|---|---|
| 4.1 | Rows were generated in pairs | permutation of income order | pairs matched vs null |
| 4.1 | Twin parity splits spenders at the extremes | Fisher exact | odds ratio |
| 4.2 | Income is unrelated to the *level* of spending | Pearson r | r with Fisher-z CI |
| 4.2 | …but strongly related to its *spread* | Brown–Forsythe; distance correlation | SD ratio (bootstrap CI); dCor |
| 4.3 | Age lowers spending | Spearman; Kruskal–Wallis over age bands | ρ (bootstrap CI); ε² |
| 4.3 | Nobody over 40 scores above 60 | Fisher exact | rate with Wilson CI; expected count |
| 4.3 | The ceiling is not income in disguise | Pearson r(age, income) | r with CI |
| 4.4 | Gender does not matter | Mann–Whitney U; Welch t | Cohen's d, Cliff's δ (bootstrap CI) |

**Assumptions.** Members are treated as independent draws. `01` showed that is not literally true (pairs),
so every p-value here is a description of *this file*, not an inference about a population. The
practical-significance judgement does not depend on the p-values.

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd()
while not (ROOT / "archive" / "Mall_Customers.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARCHIVE = ROOT / "archive"
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
REPORTS, FIGS, MODELS = ROOT / "reports", ROOT / "reports" / "figures", ROOT / "models"
for p in (RAW, PROC, REPORTS, FIGS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

KF_PATH = REPORTS / "_key_figures.json"


def _plain(v):
    if isinstance(v, dict):
        return {str(k): _plain(x) for k, x in v.items()}
    if hasattr(v, "item"):
        return v.item()
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    return v


def kf_update(**kw):
    """Append to reports/_key_figures.json — the only place the report reads numbers from."""
    kf = json.loads(KF_PATH.read_text()) if KF_PATH.exists() else {}
    kf.update({k: _plain(v) for k, v in kw.items()})
    KF_PATH.write_text(json.dumps(kf, indent=1, sort_keys=True))
    print(f"  _key_figures.json <- {', '.join(kw)}  ({len(kf)} figures total)")

from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

df = pd.read_parquet(PROC / "mall_clean.parquet")
rng = np.random.default_rng(RANDOM_STATE)
B = 2000            # bootstrap resamples
TESTS = []


def record(section, claim, test, stat, p, effect, ci=None, n=len(df)):
    TESTS.append({"section": section, "claim": claim, "test": test, "statistic": stat, "p": p,
                  "effect": effect, "ci_low": None if ci is None else ci[0],
                  "ci_high": None if ci is None else ci[1], "n": n})


def boot_ci(fn, *arrays, b=B):
    n = len(arrays[0])
    vals = [fn(*[a[idx] for a in arrays]) for idx in (rng.integers(0, n, n) for _ in range(b))]
    return tuple(np.percentile(vals, [2.5, 97.5]))


def r_ci(r, n):
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    return tuple(np.tanh([z - 1.96 * se, z + 1.96 * se]))


age, inc, sc = df.age.to_numpy(float), df.income_k.to_numpy(float), df.score.to_numpy(float)
print(f"{len(df)} members loaded")

200 members loaded


## 4.1 The file — is the pair structure chance?

If 200 incomes with this exact set of values were in random order, how many of the 100 consecutive
ID pairs would share an income by chance?

In [2]:
inc_sorted = df.sort_values("customer_id").income_k.to_numpy()
obs_pairs = int((inc_sorted[0::2] == inc_sorted[1::2]).sum())
null_pairs = np.array([(lambda v: (v[0::2] == v[1::2]).sum())(rng.permutation(inc_sorted)) for _ in range(10_000)])
p_pairs = (np.sum(null_pairs >= obs_pairs) + 1) / (len(null_pairs) + 1)
print(f"observed matching pairs: {obs_pairs} of 100")
print(f"null (random order): mean {null_pairs.mean():.1f}, max {null_pairs.max()} of 10,000 permutations")
print(f"permutation p ≤ {p_pairs:.1e}  (no permutation came close)")
record("4.1", "consecutive ID pairs share an income", "permutation (10,000)", obs_pairs, p_pairs,
       obs_pairs - null_pairs.mean())

srt = df.sort_values("customer_id")
odd, even = srt[srt.id_odd].reset_index(drop=True), srt[~srt.id_odd].reset_index(drop=True)
ext = (odd.income_k <= 40) | (odd.income_k >= 70)
tab = [[int((even.score[ext] > 50).sum()), int((even.score[ext] <= 50).sum())],
       [int((odd.score[ext] > 50).sum()), int((odd.score[ext] <= 50).sum())]]
or_, p_par = stats.fisher_exact(tab)
or_haldane = ((tab[0][0] + .5) * (tab[1][1] + .5)) / ((tab[0][1] + .5) * (tab[1][0] + .5))
print(f"\nat the income extremes ({int(ext.sum())} pairs): even twin > 50 in {tab[0][0]}, odd twin > 50 in {tab[1][0]}")
print(f"Fisher exact p = {p_par:.1e} · odds ratio (Haldane-corrected) = {or_haldane:,.0f}")
record("4.1", "twin parity splits high/low spenders at income extremes", "Fisher exact", or_, p_par,
       or_haldane, n=int(ext.sum()) * 2)

observed matching pairs: 100 of 100
null (random order): mean 1.7, max 9 of 10,000 permutations
permutation p ≤ 1.0e-04  (no permutation came close)

at the income extremes (63 pairs): even twin > 50 in 61, odd twin > 50 in 1
Fisher exact p = 4.1e-32 · odds ratio (Haldane-corrected) = 1,025


Not chance: random ordering matches fewer than two pairs in a hundred (never more than 9), the file matches all 100, and at
the income extremes parity alone separates spenders from non-spenders. The construction is a fact about
the file, which Part I of the report states before any segment size is quoted.

## 4.2 Income — level versus spread

In [3]:
r_is, p_is = stats.pearsonr(inc, sc)
record("4.2", "income predicts the level of spending", "Pearson r", r_is, p_is, r_is, r_ci(r_is, len(df)))
print(f"Pearson r(income, score) = {r_is:.3f}  95% CI [{r_ci(r_is, len(df))[0]:.2f}, {r_ci(r_is, len(df))[1]:.2f}]  p = {p_is:.2f}")

groups = [g.score.to_numpy(float) for _, g in df.groupby("income_band", observed=True)]
bf = stats.levene(*groups, center="median")
mid = df.income_band == "41–69k"


def sd_ratio(i, s):
    m = (i > 40) & (i < 70)
    return np.std(s[~m], ddof=1) / np.std(s[m], ddof=1)


ratio = sd_ratio(inc, sc)
ratio_ci = boot_ci(sd_ratio, inc, sc)
print(f"\nscore SD by band: " + ", ".join(f"{b} {g.std(ddof=1):.1f}" for b, g in zip(df.income_band.cat.categories, groups)))
print(f"Brown–Forsythe W = {bf.statistic:.1f}, p = {bf.pvalue:.1e}")
print(f"SD at the extremes ÷ SD in the middle = {ratio:.1f}×  95% CI [{ratio_ci[0]:.1f}, {ratio_ci[1]:.1f}]")
record("4.2", "income band sets the spread of spending", "Brown–Forsythe", bf.statistic, bf.pvalue, ratio, ratio_ci)


def dcor(x, y):
    a, b = np.abs(x[:, None] - x[None, :]), np.abs(y[:, None] - y[None, :])
    A = a - a.mean(0) - a.mean(1)[:, None] + a.mean()
    Bm = b - b.mean(0) - b.mean(1)[:, None] + b.mean()
    return float(np.sqrt((A * Bm).mean() / np.sqrt((A * A).mean() * (Bm * Bm).mean())))


dc = dcor(inc, sc)
dc_null = np.array([dcor(inc, rng.permutation(sc)) for _ in range(2000)])
p_dc = (np.sum(dc_null >= dc) + 1) / 2001
print(f"\ndistance correlation(income, score) = {dc:.3f}; null 95th pct {np.percentile(dc_null, 95):.3f}; "
      f"permutation p = {p_dc:.4f}")
record("4.2", "income and score are dependent (any form)", "distance correlation, permutation (2,000)", dc, p_dc, dc)

Pearson r(income, score) = 0.010  95% CI [-0.13, 0.15]  p = 0.89



score SD by band: ≤ 40k 30.2, 41–69k 7.6, ≥ 70k 33.4
Brown–Forsythe W = 140.6, p = 1.1e-38
SD at the extremes ÷ SD in the middle = 4.2×  95% CI [3.1, 5.9]



distance correlation(income, score) = 0.253; null 95th pct 0.167; permutation p = 0.0010


Both halves hold. The linear correlation is zero (CI spans roughly ±0.14), so income does not move the
*average* score. But income and score are clearly **dependent** (distance correlation 0.25, far above
anything the permutation null produces), and the dependence is in the spread: the score varies about
four times as much at the income extremes as in the middle.

## 4.3 Age — slope or ceiling?

In [4]:
rho, p_rho = stats.spearmanr(age, sc)
rho_ci = boot_ci(lambda a, s: stats.spearmanr(a, s)[0], age, sc)
record("4.3", "age is associated with lower spending", "Spearman ρ", rho, p_rho, rho, rho_ci)
print(f"Spearman ρ(age, score) = {rho:.2f}  95% CI [{rho_ci[0]:.2f}, {rho_ci[1]:.2f}]  p = {p_rho:.1e}")

kw = stats.kruskal(*[g.score for _, g in df.groupby("age_group", observed=True)])
k = df.age_group.nunique()
eps2 = (kw.statistic - k + 1) / (len(df) - k)
record("4.3", "spending differs across age bands", "Kruskal–Wallis", kw.statistic, kw.pvalue, eps2)
print(f"Kruskal–Wallis H = {kw.statistic:.1f}, p = {kw.pvalue:.1e}, ε² = {eps2:.2f}")

ct = pd.crosstab(df.age_over_40, df.score > 60)
_, p_ceil = stats.fisher_exact(ct.values)
n_old, n_old_hi = int(df.age_over_40.sum()), int(ct.loc[True, True]) if True in ct.columns else 0
rate_ci = proportion_confint(n_old_hi, n_old, method="wilson")
base_hi = (df.score > 60).mean()
expected = n_old * base_hi
record("4.3", "nobody over 40 scores above 60", "Fisher exact", n_old_hi, p_ceil, n_old_hi / n_old, rate_ci, n=len(df))
print(f"\nover-40s scoring > 60: {n_old_hi} of {n_old};  Wilson 95% CI for the rate [{rate_ci[0]:.1%}, {rate_ci[1]:.1%}]")
print(f"expected if age were irrelevant: {expected:.0f} (overall rate {base_hi:.0%})   Fisher p = {p_ceil:.1e}")

r_ai, p_ai = stats.pearsonr(age, inc)
record("4.3", "age and income are related (confound check)", "Pearson r", r_ai, p_ai, r_ai, r_ci(r_ai, len(df)))
print(f"\nPearson r(age, income) = {r_ai:.2f}  95% CI [{r_ci(r_ai, len(df))[0]:.2f}, {r_ci(r_ai, len(df))[1]:.2f}]  → not a confound")
by_band = df[df.age_over_40].groupby("income_band", observed=True).score.max()
print("max score of over-40s, by income band:", by_band.to_dict())

Spearman ρ(age, score) = -0.34  95% CI [-0.45, -0.23]  p = 5.7e-07
Kruskal–Wallis H = 37.7, p = 1.3e-07, ε² = 0.17

over-40s scoring > 60: 0 of 78;  Wilson 95% CI for the rate [0.0%, 4.7%]
expected if age were irrelevant: 24 (overall rate 31%)   Fisher p = 1.2e-17

Pearson r(age, income) = -0.01  95% CI [-0.15, 0.13]  → not a confound
max score of over-40s, by income band: {'≤ 40k': 55, '41–69k': 60, '≥ 70k': 39}


The ceiling is the stronger description. Where an age-blind file would put about two dozen over-40s
above 60, the file puts **none**; the upper end of the CI for that rate is under 5%. It holds in every
income band, and age is independent of income, so the ceiling is not a disguised income effect. The
Spearman slope (−0.34) is real but describes the ceiling badly: under 40, scores span 1–99.

## 4.4 Gender

In [5]:
f = df.loc[df.gender == "Female", "score"].to_numpy(float)
m = df.loc[df.gender == "Male", "score"].to_numpy(float)
mw = stats.mannwhitneyu(f, m)
welch = stats.ttest_ind(f, m, equal_var=False)


def cohen_d(x, y):
    sp = np.sqrt(((len(x) - 1) * x.var(ddof=1) + (len(y) - 1) * y.var(ddof=1)) / (len(x) + len(y) - 2))
    return (x.mean() - y.mean()) / sp


d = cohen_d(f, m)
d_ci = tuple(np.percentile([cohen_d(rng.choice(f, len(f)), rng.choice(m, len(m))) for _ in range(B)], [2.5, 97.5]))
cliff = 2 * mw.statistic / (len(f) * len(m)) - 1
record("4.4", "women and men differ in spending", "Mann–Whitney U", mw.statistic, mw.pvalue, cliff)
record("4.4", "women and men differ in mean spending", "Welch t", welch.statistic, welch.pvalue, d, d_ci)
print(f"mean score: female {f.mean():.1f}, male {m.mean():.1f}  (gap {f.mean() - m.mean():.1f} points)")
print(f"Mann–Whitney p = {mw.pvalue:.2f} · Cliff's δ = {cliff:.2f}")
print(f"Welch t p = {welch.pvalue:.2f} · Cohen's d = {d:.2f}  95% CI [{d_ci[0]:.2f}, {d_ci[1]:.2f}]")

mean score: female 51.5, male 48.5  (gap 3.0 points)
Mann–Whitney p = 0.57 · Cliff's δ = 0.05
Welch t p = 0.42 · Cohen's d = 0.12  95% CI [-0.19, 0.41]


The CI for Cohen's d includes zero and stays within "small" (|d| < 0.5) at both ends. Gender is not a
driver of the score in this file.

## 4.5 Multiple-comparison correction and the driver summary

In [6]:
tests = pd.DataFrame(TESTS)
tests["p_bh"] = multipletests(tests.p, method="fdr_bh")[1]
tests["significant_bh"] = tests.p_bh < 0.05
tests.to_csv(REPORTS / "_tests.csv", index=False)
print(f"{len(tests)} tests · {tests.significant_bh.sum()} significant after Benjamini–Hochberg (q < 0.05)\n")
tests[["section", "claim", "test", "effect", "ci_low", "ci_high", "p", "p_bh", "significant_bh"]].round(4)

11 tests · 7 significant after Benjamini–Hochberg (q < 0.05)



,section,claim,test,effect,ci_low,ci_high,p,p_bh,significant_bh
0,4.1,consecutive ID pairs share an income,"permutation (10,000)",98.2564,NaN,NaN,0.0001,0.0002,True
1,4.1,twin parity splits high/low spenders at income...,Fisher exact,1025.0000,NaN,NaN,0.0000,0.0000,True
2,4.2,income predicts the level of spending,Pearson r,0.0099,-0.1290,0.1484,0.8893,0.8893,False
3,4.2,income band sets the spread of spending,Brown–Forsythe,4.2167,3.0526,5.9254,0.0000,0.0000,True
4,4.2,income and score are dependent (any form),"distance correlation, permutation (2,000)",0.2532,NaN,NaN,0.0010,0.0016,True
5,4.3,age is associated with lower spending,Spearman ρ,-0.3449,-0.4496,-0.2275,0.0000,0.0000,True
6,4.3,spending differs across age bands,Kruskal–Wallis,0.1730,NaN,NaN,0.0000,0.0000,True
7,4.3,nobody over 40 scores above 60,Fisher exact,0.0000,0.0000,0.0469,0.0000,0.0000,True
8,4.3,age and income are related (confound check),Pearson r,-0.0124,-0.1509,0.1266,0.8617,0.8893,False
9,4.4,women and men differ in spending,Mann–Whitney U,0.0468,NaN,NaN,0.5713,0.6982,False


In [7]:
row = lambda c: tests.set_index("claim").loc[c]
kf_update(
    n_tests=len(tests), n_significant_bh=int(tests.significant_bh.sum()),
    pairs_null_mean=float(null_pairs.mean()), pairs_null_max=int(null_pairs.max()), pairs_perm_p=float(p_pairs),
    parity_fisher_p=float(p_par),
    r_income_score=float(r_is), r_income_score_ci=list(r_ci(r_is, len(df))), p_income_score=float(p_is),
    sd_ratio_extremes=float(ratio), sd_ratio_ci=list(ratio_ci), bf_p=float(bf.pvalue),
    dcor_income_score=float(dc), dcor_p=float(p_dc), dcor_null_p95=float(np.percentile(dc_null, 95)),
    rho_age_score=float(rho), rho_age_score_ci=list(rho_ci), p_rho_age=float(p_rho),
    kw_age_eps2=float(eps2), kw_age_p=float(kw.pvalue),
    ceiling_rate_ci=list(rate_ci), ceiling_expected=float(expected), ceiling_p=float(p_ceil),
    r_age_income=float(r_ai), r_age_income_ci=list(r_ci(r_ai, len(df))),
    gender_d=float(d), gender_d_ci=list(d_ci), gender_mw_p=float(mw.pvalue), gender_welch_p=float(welch.pvalue),
    gender_cliff=float(cliff),
    gender_p_bh=float(row("women and men differ in mean spending").p_bh),
    income_level_p_bh=float(row("income predicts the level of spending").p_bh),
)

  _key_figures.json <- n_tests, n_significant_bh, pairs_null_mean, pairs_null_max, pairs_perm_p, parity_fisher_p, r_income_score, r_income_score_ci, p_income_score, sd_ratio_extremes, sd_ratio_ci, bf_p, dcor_income_score, dcor_p, dcor_null_p95, rho_age_score, rho_age_score_ci, p_rho_age, kw_age_eps2, kw_age_p, ceiling_rate_ci, ceiling_expected, ceiling_p, r_age_income, r_age_income_ci, gender_d, gender_d_ci, gender_mw_p, gender_welch_p, gender_cliff, gender_p_bh, income_level_p_bh  (78 figures total)


### What survived, and what did not

| Stage 0 hypothesis | Verdict | Evidence |
|---|---|---|
| Income alone does not drive spending | **Refined** | True for the level (r ≈ 0), false for the spread (SD ratio ~4×, dCor 0.25) |
| Age has a modest negative relationship | **Refined** | Not a slope but a ceiling: 0 of 78 over-40s above 60 |
| Gender is nearly irrelevant | **Confirmed** | d ≈ 0.12, CI includes 0; not significant after correction |
| The file is a clean snapshot of a member base | **Rejected** | Generated in income-sorted pairs; parity splits the corners |

## Stage 5a — Gate Checklist

- [x] Tests chosen by data shape: permutation and distance correlation where the dependence is
      non-linear; rank tests where scores are bounded; Fisher exact for the zero cell
- [x] Assumptions stated (independence is violated by construction; results describe this file)
- [x] Effect sizes with CIs for every claim (bootstrap, Fisher-z, Wilson)
- [x] Benjamini–Hochberg applied across all tests (`reports/_tests.csv`)
- [x] Results interpreted in business terms (targeting by score at the income extremes; age 40 as a
      boundary; no gender targeting)